In [1]:
# Importar librerías
import pandas as pd
from src.config import data_folder

%load_ext autoreload
%autoreload 2
from src.extract import *

In [2]:
# Extraer datos de simFin y generar lista de tickers según los criterios
df_simfin_unfiltered = extraer_simfin()
tickers_universe_list = generar_universo_tickers(df_simfin_unfiltered)

# Se agrega columna 'FinancialsSource' para indicar que los datos provienen de simFin
df_simfin_unfiltered['FinancialsSource'] = 'simFin'

print("Datos extraidos de simFin, dimensiones:", df_simfin_unfiltered.shape)

Dataset "us-income-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Dataset "us-income-banks-quarterly" on disk (5 days old).
- Loading from disk ... Done!
Dataset "us-balance-banks-quarterly" on disk (5 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-banks-quarterly" on disk (5 days old).
- Loading from disk ... Done!
Dataset "us-income-insurance-quarterly" on disk (5 days old).
- Loading from disk ... Done!
Dataset "us-balance-insurance-quarterly" on disk (5 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-insurance-quarterly" on disk (5 days old).
- Loading from disk ... Done!
Datos extraidos de simFin, dimensiones: (50230, 83)


In [3]:
# Extraer precios de los tickers y del índice SPY (demora unos minutos)
df_prices = extraer_precios(tickers_universe_list)

# Se extrae precio del Índice de Mercado para usar en cálculos y se guarda en fichero 
df_index = extraer_precios(['SPY'])
df_index.to_parquet(f"{data_folder}/market_index.parquet")

print("Extraídos precios del universo de tickers y del indice.")
print("Dimensiones de df_precios:", df_prices.shape)
print("Dimensiones de df_index:", df_index.shape)

Iniciando descarga masiva para 549 tickers únicos...


[*                      2%                       ]  13 of 549 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: RYI"}}}
[*********************100%***********************]  549 of 549 completed

27 Failed downloads:
['RYI', 'MDC', 'X', 'TM-28', 'GMS', 'HOUS', 'K', 'FLT', 'CCH', 'SAVE', 'JWN', 'SQ', 'DISH', 'IPG', 'FL', 'JNPR', 'HES', 'HBI', 'WBA', 'SPR', 'PDCO', 'SKX', 'FI', 'APY', 'QRTEA', 'TPX']: YFPricesMissingError('possibly delisted; no price data found  (period=6y) (Yahoo error = "No data found, symbol may be delisted")')
['QVC']: YFPricesMissingError('possibly delisted; no price data found  (period=6y)')


Reestructurando los datos...
Extracción completada.
Iniciando descarga masiva para 1 tickers únicos...


[*********************100%***********************]  1 of 1 completed

Reestructurando los datos...
Extracción completada.
Extraídos precios del universo de tickers y del indice.
Dimensiones de df_precios: (13026, 5)
Dimensiones de df_index: (25, 5)


In [4]:
# Actualizar el universo de tickers con los que se obtuvieron precios
tickers_list_updated = df_prices['Ticker'].unique().tolist()

print("Actualizado el universo a tickers con precios disponibles en yfinance.")
print("Cantidad de tickers actualizado:", len(tickers_list_updated))

Actualizado el universo a tickers con precios disponibles en yfinance.
Cantidad de tickers actualizado: 522


In [5]:
# Filtrar tickers en datos de simFin
df_simfin = df_simfin_unfiltered[df_simfin_unfiltered['Ticker'].isin(tickers_list_updated)]
df_simfin.shape

(9937, 83)

Obtener lista de tickers del S&P 500 desde el fichero constituents.csv:

* De dicho fichero se obtiene para los tickers que pertenecen al índice, la fecha en la que fueron agregados con la columna `DataAdded`.
* Si no existe el fichero, lo descarga desde GitHub.
* Para actualizar la lista de componentes, cambiar `force_update` a True, ejecutar y volver a dejar en False.

In [6]:
# Obtener constituents del indice S&P 500
ruta_sp500 = descargar_constituents_sp(force_update=False) 

# Cargar y limpiar datos de constituents
print("Cargando los tickers del S&P 500.")
df_tickers_sp_raw = pd.read_csv(ruta_sp500)
df_tickers_sp = limpiar_constituents_sp(df_tickers_sp_raw)
print("Cantidad de tickers:", df_tickers_sp['Ticker'].nunique())
df_tickers_sp.info()

Usando archivo constituents.csv local.
Cargando los tickers del S&P 500.
Cantidad de tickers: 503
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Ticker     503 non-null    object
 1   DateAdded  503 non-null    object
dtypes: object(2)
memory usage: 8.0+ KB


In [7]:
# Unir df_prices y df_tickers
df_merged = pd.merge(
    df_prices,
    df_tickers_sp,
    on= 'Ticker',
    how= 'left'
)
print("Unidos precios con DateAdded del indice S&P500, dimensiones del dataset:", df_merged.shape)
df_merged.info()

Unidos precios con DateAdded del indice S&P500, dimensiones del dataset: (13026, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13026 entries, 0 to 13025
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       13026 non-null  datetime64[ns]
 1   Ticker     13026 non-null  object        
 2   Close      13026 non-null  float64       
 3   Open       13026 non-null  float64       
 4   Volume     13026 non-null  float64       
 5   DateAdded  7565 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(2)
memory usage: 610.7+ KB


In [8]:
# Extraer info de sector e industria (demora unos minutos)
df_info = extraer_info(tickers_list_updated)
df_info.info()

Error con MKSI: Failed to perform, curl: (28) Operation timed out after 636608 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Ticker    521 non-null    object
 1   Sector    521 non-null    object
 2   Industry  521 non-null    object
dtypes: object(3)
memory usage: 12.3+ KB


In [9]:
# Se unen los dataframes
df_with_info = pd.merge(
    df_merged,
    df_info,
    on= 'Ticker',
    how= 'left'
)
df_with_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13026 entries, 0 to 13025
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       13026 non-null  datetime64[ns]
 1   Ticker     13026 non-null  object        
 2   Close      13026 non-null  float64       
 3   Open       13026 non-null  float64       
 4   Volume     13026 non-null  float64       
 5   DateAdded  7565 non-null   object        
 6   Sector     13001 non-null  object        
 7   Industry   13001 non-null  object        
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 814.3+ KB


## Extraer datos financieros de yfinance (últimos 4 reportes trimestrales)
`yfinance` no incluye la fecha de publicación, sólo la fecha de los Estados Financieros. 

Para evitar el "LookAhead" bias, se ofrecen 2 alternativas con el parametro `aproximar_fechas`:

*aproximar_fechas = True*: 
* Se estima la fecha de publicación con una regla de 30 días por defecto (editable en src/config.py). 
* *Demora de ejecución* de aproximadamente 10 minutos.

*aproximar_fechas = False*: 
* Obtiene la fecha real de publicación usando yf.earnings_dates. 
* Se aplica la regla de 30 días sólo si no se encuentra. 
* Puede demorar 30 minutos o más.

In [ ]:
# Extraer datos financieros de yfinance: ultimos 4 trimestres
print("Extrayendo datos financieros de yfinance, demora varios minutos.")
df_yfinance, tickers_sin_datos = extraer_financials(tickers_list_updated, aproximar_fechas = False)

# Se agrega columna 'FinancialsSource' para indicar que los datos provienen de yfinance
df_yfinance['FinancialsSource'] = 'yfinance'
df_yfinance.info()

Extrayendo datos financieros de yfinance, demora varios minutos.
Aviso: Error obteniendo fechas reales para BURL. Usando estimación. (Failed to perform, curl: (28) Operation timed out after 540076 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.)
Sin datos financieros trimestrales para CNLPL


CTA-PB: No earnings dates found, symbol may be delisted
RSTRF: No earnings dates found, symbol may be delisted


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2084 entries, 0 to 2083
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Date                           2084 non-null   datetime64[ns]
 1   Total Revenue                  2080 non-null   float64       
 2   Gross Profit                   1951 non-null   float64       
 3   Operating Income               1965 non-null   float64       
 4   Net Income                     2080 non-null   float64       
 5   EBITDA                         1965 non-null   float64       
 6   Basic Average Shares           2039 non-null   float64       
 7   Cash And Cash Equivalents      2071 non-null   float64       
 8   Current Debt                   1560 non-null   float64       
 9   Long Term Debt                 1952 non-null   float64       
 10  Total Debt                     2057 non-null   float64       
 11  Stockholders Equi

In [14]:
# Definir columnas a mantener en simFin para que coincidan y estandarizar antes de unir
cols_yfinance = df_yfinance.columns.to_list()
df_simfin_clean = estandarizar_simfin(df_simfin, cols_yfinance)

df_financials_completo = unir_financials(df_yfinance, df_simfin_clean)
df_financials_completo.info()

Se han encontrado 35 filas con Ticker y Date solapados (presentes en ambas fuentes).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11688 entries, 0 to 11687
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Ticker                         11688 non-null  object        
 1   Date                           11672 non-null  datetime64[ns]
 2   Total Revenue                  11668 non-null  float64       
 3   Gross Profit                   10681 non-null  float64       
 4   Operating Income               11553 non-null  float64       
 5   Net Income                     11668 non-null  float64       
 6   EBITDA                         1940 non-null   float64       
 7   Basic Average Shares           11620 non-null  float64       
 8   Cash And Cash Equivalents      11629 non-null  float64       
 9   Current Debt                   8870 non-null   float64       
 1

In [15]:
# Unir dataframe de precios con datos financieros
df_final = pd.merge(
    df_with_info, 
    df_financials_completo, 
    on=['Date', 'Ticker'],
    how='left'
)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13026 entries, 0 to 13025
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Date                           13026 non-null  datetime64[ns]
 1   Ticker                         13026 non-null  object        
 2   Close                          13026 non-null  float64       
 3   Open                           13026 non-null  float64       
 4   Volume                         13026 non-null  float64       
 5   DateAdded                      7565 non-null   object        
 6   Sector                         13001 non-null  object        
 7   Industry                       13001 non-null  object        
 8   Total Revenue                  11523 non-null  float64       
 9   Gross Profit                   10540 non-null  float64       
 10  Operating Income               11408 non-null  float64       
 11  Net Income     

In [16]:
# Limpieza final
df_final_clean = limpieza_final(df_final)
df_final_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11408 entries, 0 to 11407
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         11408 non-null  datetime64[ns]
 1   Ticker                       11408 non-null  object        
 2   Close                        11408 non-null  float64       
 3   Open                         11408 non-null  float64       
 4   Volume                       11408 non-null  float64       
 5   DateAdded                    6720 non-null   object        
 6   Sector                       11386 non-null  object        
 7   Industry                     11386 non-null  object        
 8   TotalRevenue                 11408 non-null  float64       
 9   GrossProfit                  10540 non-null  float64       
 10  OperatingIncome              11408 non-null  float64       
 11  NetIncome                    11408 non-nu

In [17]:
# Se quitan del dataset los tickers sin datos financieros de yfinance
tickers_unicos = set(tickers_sin_datos) # se convierte en set primero por si hay tickers duplicados
df_final_clean = df_final_clean[~df_final_clean['Ticker'].isin(tickers_unicos)].reset_index(drop=True)

tickers_finales = actualizar_fichero_tickers(df_final_clean)

print(f"Eliminados tickers sin datos financieros en yfinance, quedan {len(tickers_finales)} tickers.")
print("Dimensiones del dataset actualizado:", df_final_clean.shape)

Eliminados tickers sin datos financieros en yfinance, quedan 516 tickers.
Dimensiones del dataset actualizado: (11389, 29)


In [18]:
df_final_clean.head()

,Date,Ticker,Close,Open,Volume,DateAdded,Sector,Industry,TotalRevenue,GrossProfit,...,TotalAssets,CurrentAssets,CurrentLiabilities,OperatingCashFlow,InvestingCashFlow,FinancingCashFlow,FreeCashFlow,CapitalExpenditure,DepreciationAndAmortization,FinancialsSource
0,2020-12-01,A,117.513367,113.190972,101905300.0,2000-06-05,Healthcare,Diagnostics & Research,1.261000e+09,669000000.0,...,9.546000e+09,3.245000e+09,1.314000e+09,290000000.0,-32000000.0,-231000000.0,NaN,-24000000.0,NaN,simFin
1,2021-03-01,A,133.191986,118.651080,101440700.0,2000-06-05,Healthcare,Diagnostics & Research,1.483000e+09,788000000.0,...,9.627000e+09,3.415000e+09,1.467000e+09,377000000.0,-27000000.0,-269000000.0,NaN,-27000000.0,NaN,simFin
2,2021-06-01,A,169.454529,134.398964,114519900.0,2000-06-05,Healthcare,Diagnostics & Research,1.548000e+09,838000000.0,...,9.674000e+09,3.483000e+09,1.687000e+09,238000000.0,-42000000.0,-316000000.0,NaN,-41000000.0,NaN,simFin
3,2021-09-01,A,145.918365,169.261436,93038700.0,2000-06-05,Healthcare,Diagnostics & Research,1.525000e+09,817000000.0,...,1.039800e+10,3.514000e+09,1.758000e+09,472000000.0,-587000000.0,166000000.0,NaN,-31000000.0,NaN,simFin
4,2021-12-01,A,126.213455,146.313107,116125700.0,2000-06-05,Healthcare,Diagnostics & Research,1.586000e+09,852000000.0,...,1.049100e+10,3.632000e+09,1.724000e+09,334000000.0,-61000000.0,-222000000.0,NaN,-54000000.0,NaN,simFin


In [19]:
# Guardar datos extraidos en fichero raw_data
df_final_clean.to_parquet(f"{data_folder}/raw_data.parquet")
print(f"Extraccion finalizada.\nFichero 'raw_data.parquet' guardado en la carpeta {data_folder}")

Extraccion finalizada.
Fichero 'raw_data.parquet' guardado en la carpeta data
